
## Drei Judges, ein Muster

Wir bauen einen Judge pro Frage:

- **Faithfulness** — wird die Antwort durch den Kontext gestützt?
- **Relevance** — beantwortet die Antwort die Frage?
- **Correctness** — stimmt die Antwort mit der Referenz überein?

Unterschiede:

- **Answer Relevancy:** Beantwortet die Antwort die richtige Frage? Antwort <--> sollte zur Frage passen.
- **Answer Correctness:** Ist die Antwort auch tatsächlich richtig? Antwort <--> sollte zu Referenz passen.
- **Faithfulness:** Ist die Antwort durch die bereitgestellten Dokumente belegt? Antwort <--> sollte im Kontext stehen.

In [2]:
import os
from dotenv import load_dotenv
from langchain_openrouter import ChatOpenRouter
from pydantic import SecretStr
from pydantic import BaseModel, Field
from rag_tutorial.rag_utils import create_rag_system, Judgement, build_rag_chain, create_judge_llm, make_judges, print_wrapped
import pandas as pd

c:\Artificial Intelligence\rag_tutorial\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
load_dotenv()
api_key = SecretStr(os.getenv("OPENROUTER_API_KEY", ""))

In [4]:
judge_llm = ChatOpenRouter(
    api_key=api_key,
    model="ibm-granite/granite-4.2-8b",
    temperature=0,
     reasoning={"effort": "minimal"},
)

judge = judge_llm.with_structured_output(Judgement)


## Judge 1: Faithfulness

- Wird jede Aussage in der Antwort durch den Kontext gestützt?
- Eine ungedeckte Aussage ist oft **falsch** — eine **Halluzination**, der
  gefährlichste RAG-Fehler
- Er erwischt auch *korrekte* Aussagen, die der Kontext nie gemacht hat
  (eigenes Wissen) — auch die wollen wir von einem Nur-Dokumente-System nicht
- Im (englischen) Prompt heißt das Kriterium *grounded* — das übliche Wort
  in Judge-Prompts

In [5]:
FAITHFULNESS_PROMPT = """\
You check whether an answer is grounded in the given context.
The answer must only use information found in or reasonably inferred from the context.
If it contains any claim not supported by the context, the verdict is False.

Context:
{context}

Answer:
{answer}"""

In [6]:
context = (
    "Overfitting tritt auf, wenn ein Modell die Trainingsdaten auswendig lernt. "
    "Regularisierung hilft dagegen."
)

answer = "Overfitting bedeutet, dass das Modell die Trainingsdaten auswendig lernt."

In [7]:
def judge_faithfulness(answer, context):
    return judge.invoke(
        FAITHFULNESS_PROMPT.format(context=context, answer=answer)
    )

In [8]:
result = judge_faithfulness(answer, context)
(result.explanation, result.verdict)

('The answer accurately paraphrases the context without adding unsupported claims.',
 True)

In [9]:
# Answer with halutination
answer = "Overfitting wird durch eine zu niedrige Lernrate verursacht."
result = judge_faithfulness(answer, context)
(result.explanation, result.verdict)

('The answer claims overfitting is caused by a low learning rate, which is not mentioned in the context.',
 False)


## Judge 2: Relevanz

- Geht die Antwort tatsächlich auf die Frage ein?
- Erkennt **themenfremde** Antworten (benötigt keinen Kontext oder Referenzwert)

In [10]:
RELEVANCE_PROMPT = """\
You check whether an answer addresses the question that was asked.
A relevant answer responds to the question, even if it turns out to be wrong.
An answer about a different topic has the verdict False.

Question:
{question}

Answer:
{answer}"""


In [11]:
def judge_relevance(question, answer):
    return judge.invoke(
        RELEVANCE_PROMPT.format(question=question, answer=answer)
    )

In [12]:
judge_relevance(
    "Wie wird ein neuronales Netz trainiert?",
    "Durch Backpropagation und Gradient Descent.",
)

Judgement(verdict=True, explanation='The answer directly addresses the question about how a neural network is trained.')

In [13]:
# Eine themenfremde Antwort
judge_relevance(
    "Wie wird ein neuronales Netz trainiert?",
    "Random Forests kombinieren viele Decision Trees.",
)

Judgement(verdict=False, explanation='The answer discusses Random Forests, which are not relevant to training a neural network.')


## Judge 3: Korrektheit

- Stimmt die Antwort mit der **Referenz**antwort überein?
- Geringfügige Formulierungsunterschiede sind in Ordnung — wir vergleichen die *Bedeutung*

In [14]:
CORRECTNESS_PROMPT = """\
You compare a generated answer with a reference answer.
The verdict is True if the generated answer is consistent with the reference.
Minor wording differences are acceptable.
If the reference says the information is not available, the generated answer
should indicate the same.

Question:
{question}

Reference answer:
{reference}

Generated answer:
{answer}"""

In [15]:
def judge_correctness(question, reference, answer):
    return judge.invoke(
        CORRECTNESS_PROMPT.format(
            question=question, reference=reference, answer=answer
        )
    )

In [16]:
result = judge_correctness(
    "Wofür wird Mean Squared Error verwendet?",
    "Als Loss-Funktion bei der Linearen Regression.",
    "MSE ist die Loss-Funktion der linearen Regression.",
)

(result.explanation, result.verdict)

('The generated answer correctly identifies MSE as the loss function for linear regression, consistent with the reference answer.',
 True)

### Workshop: LLM Judge

In [34]:
import pandas as pd

In [35]:
LANGUAGE = "de"
_, _, vectorstore, llm, judge_llm = create_rag_system(
    k=2, collection_name="ml_docs_run", language=LANGUAGE
)

In [19]:
judge_faithfulness, judge_relevance, judge_correctness = make_judges(judge_llm)

In [20]:
test_set = [
    {"question": "Was ist Overfitting?",
     "reference": "Overfitting tritt auf, wenn ein Modell die Trainingsdaten "
     "auswendig lernt und nicht auf neue Daten generalisiert."},
    {"question": "Wie werden Neuronale Netze trainiert?",
     "reference": "Neuronale Netze werden durch Backpropagation und Gradient "
     "Descent trainiert."},
    {"question": "Was ist der Vorteil von Random Forests?",
     "reference": "Random Forests kombinieren viele Decision Trees und sind "
     "oft genauer als einzelne Modelle."},
    {"question": "Wie hilft RAG bei LLMs?",
     "reference": "RAG hilft LLMs, präziser mit spezifischem Wissen zu antworten."},
    {"question": "Was ist der Unterschied zwischen Bagging und Boosting?",
     "reference": "Diese Information ist nicht in den Dokumenten enthalten."},
]

In [22]:
def evaluate(k):
    retriever, chain = build_rag_chain(vectorstore, llm, k, language=LANGUAGE)
    rows = []
    for item in test_set:
        q, ref = item["question"], item["reference"]
        context = "\n\n".join(d.page_content for d in retriever.invoke(q))
        answer = chain.invoke(q)
        rows.append({
            "question": q,
            "answer": answer,
            "faithful": judge_faithfulness(answer, context).verdict,
            "relevant": judge_relevance(q, answer).verdict,
            "correct": judge_correctness(q, ref, answer).verdict,
        })
    return pd.DataFrame(rows)

In [23]:
results_k2 = evaluate(k=2)

In [24]:
results_k2

,question,answer,faithful,relevant,correct
0,Was ist Overfitting?,"Overfitting tritt auf, wenn ein Modell die Tra...",True,True,False
1,Wie werden Neuronale Netze trainiert?,Training erfolgt durch Backpropagation und Gra...,True,True,True
2,Was ist der Vorteil von Random Forests?,Random Forests kombinieren viele Decision Tree...,True,True,True
3,Wie hilft RAG bei LLMs?,"RAG hilft LLMs, präziser mit spezifischem Wiss...",True,True,True
4,Was ist der Unterschied zwischen Bagging und B...,Diese Information ist nicht in meinen Dokument...,False,False,True


In [25]:
results_k2[["question", "faithful", "relevant", "correct"]]

,question,faithful,relevant,correct
0,Was ist Overfitting?,True,True,False
1,Wie werden Neuronale Netze trainiert?,True,True,True
2,Was ist der Vorteil von Random Forests?,True,True,True
3,Wie hilft RAG bei LLMs?,True,True,True
4,Was ist der Unterschied zwischen Bagging und B...,False,False,True


In [26]:
verdict_cols = ["faithful", "relevant", "correct"]

In [27]:
results_k2[verdict_cols].mean()

faithful    0.8
relevant    0.8
correct     0.8
dtype: float64


## Der Verbesserungszyklus

- **Eine Sache** ändern, neu bewerten, vergleichen
- Hier: mehr Kontext abrufen (`k=4` statt `k=2`)
- Mehr Kontext kann die Faithfulness verbessern — oder Rauschen hinzufügen

In [28]:
results_k4 = evaluate(k=4)
results_k4[verdict_cols].mean()

faithful    0.8
relevant    0.8
correct     0.8
dtype: float64

In [29]:
comparison = pd.DataFrame({
    "k=2": results_k2[verdict_cols].mean(),
    "k=4": results_k4[verdict_cols].mean(),
})

In [30]:
comparison

,k=2,k=4
faithful,0.8,0.8
relevant,0.8,0.8
correct,0.8,0.8



## Eine zweite Meinung: das Judge-Modell austauschen

- Der Judge muss nicht das RAG-Modell sein — der Austausch ist **eine Zeile**
- Ein anderes Modell umgeht die Selbstbevorzugung und liefert eine unabhängige Sicht

In [31]:
judge2_llm = create_judge_llm("openai/gpt-4.1-nano")
_, judge_relevance2, _ = make_judges(judge2_llm)

In [42]:
results_k2["relevant_judge2"] = [
    judge_relevance2(q, a).verdict
    for q, a in zip(results_k2["question"], results_k2["answer"])
]

In [43]:
results_k2

,question,answer,faithful,relevant,correct,relevant_judge2,judge
0,Was ist Overfitting?,"Overfitting tritt auf, wenn ein Modell die Tra...",True,True,False,True,True
1,Wie werden Neuronale Netze trainiert?,Training erfolgt durch Backpropagation und Gra...,True,True,True,True,True
2,Was ist der Vorteil von Random Forests?,Random Forests kombinieren viele Decision Tree...,True,True,True,True,True
3,Wie hilft RAG bei LLMs?,"RAG hilft LLMs, präziser mit spezifischem Wiss...",True,True,True,True,True
4,Was ist der Unterschied zwischen Bagging und B...,Diese Information ist nicht in meinen Dokument...,False,False,True,False,False


In [ ]:
results_k2[["question", "relevant", "relevant_judge2"]]

,question,relevant,relevant_judge2
0,Was ist Overfitting?,True,True
1,Wie werden Neuronale Netze trainiert?,True,True
2,Was ist der Vorteil von Random Forests?,True,True
3,Wie hilft RAG bei LLMs?,True,True
4,Was ist der Unterschied zwischen Bagging und B...,False,False


: 